# Representative UMAP of BICAN April 2026 freeze data

Run on a laptop

Stephen Fleming

2026.09.25

## Specify data location

This is a data extract created by Cellarium Nexus, containing 3.86M cells

In [35]:
h5ad_base_path = "gs://cellarium-nexus-file-system-3293a8/pipeline/data-extracts/bican_202604_fullextract/extract_files"

In [36]:
# can choose whether to run on CPU or use MPS

ACCELERATOR = "cpu"  # should be in ["cpu", "mps", "cuda", "auto"]

## Boilerplate to be factored out into cellarium API

I am envisioning making a very minimal API more like scanpy, that would easily interoperate with scanpy. For now, I have boilerplate code in this notebook.

In [37]:
# remove verbose logging from C
import os
os.environ["GRPC_VERBOSITY"] = "NONE" # NONE is even quieter than ERROR
os.environ["GLOG_minloglevel"] = "3"  # 3 is FATAL only
os.environ["GRPC_ENABLE_FORK_SUPPORT"] = "0"
os.environ["PYTHONWARNINGS"] = "ignore::FutureWarning"

In [38]:
from typing import Callable, Literal
import multiprocessing

import pandas as pd
import torch

from cellarium.ml.api import (
    h5ad_paths_from_google_bucket,
    get_h5ad_files_limits,
    datamodule_var_names_g,
    datamodule_obs_nunique,
)

from cellarium.ml import CellariumAnnDataDataModule, CellariumModule
from cellarium.ml.data import DistributedAnnDataCollection
from cellarium.ml.utilities.data import AnnDataField, keep_sparse, to_float_column, to_torch_sparse_csr, densify
from cellarium.ml.models import OnePassMeanVarStd, HVGSeuratV3, IncrementalPCA, StreamingPlaidGeometricSketch
from cellarium.ml.transforms import Filter, Densify, NormalizeTotal, Log1p, ZScore

In [39]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [40]:
h5ad_paths = h5ad_paths_from_google_bucket(h5ad_base_path)
h5ad_paths[:3]

['gs://cellarium-nexus-file-system-3293a8/pipeline/data-extracts/bican_202604_fullextract/extract_files/extract_000000.h5ad',
 'gs://cellarium-nexus-file-system-3293a8/pipeline/data-extracts/bican_202604_fullextract/extract_files/extract_000001.h5ad',
 'gs://cellarium-nexus-file-system-3293a8/pipeline/data-extracts/bican_202604_fullextract/extract_files/extract_000002.h5ad']

In [ ]:
def get_datamodule(
    h5ad_paths, 
    obs_columns: dict[str, Callable] = {}, 
    batch_size=10000, 
    shuffle=True, 
    train_size=1.0, 
    stage="fit",
    total_mrna_umis_key="n_counts",
    nexus_extract_uniform_sizes=True,
):
    datamodule = CellariumAnnDataDataModule(
        dadc=DistributedAnnDataCollection(
            filenames=h5ad_paths,
            limits=get_h5ad_files_limits(h5ad_paths, nexus_extract_uniform_sizes=nexus_extract_uniform_sizes),
            obs_columns_to_validate=list(obs_columns.values()),
            max_cache_size=2,
        ),
        batch_keys={
            "x_ng": AnnDataField(
                attr="X", 
                convert_fn=to_torch_sparse_csr if ACCELERATOR not in ["mps", "auto"] else densify,
            ),
            "var_names_g": AnnDataField(attr="var_names"),
            "obs_names_n": AnnDataField(attr="obs_names"),
            "total_mrna_umis_n": AnnDataField(attr="obs", key=total_mrna_umis_key, convert_fn=torch.FloatTensor),
            **{col: AnnDataField(attr="obs", key=col, convert_fn=fn) for col, fn in obs_columns.items()},
        },
        batch_size=batch_size,
        shuffle=shuffle,
        train_size=train_size,
        num_workers=4,
        prefetch_factor=2,
    )

    datamodule.setup(stage=stage)
    return datamodule

In [42]:
# this can take a few mins. it has to read n_obs (quickly) from every h5ad file

datamodule = get_datamodule(h5ad_paths)
datamodule.dadc

Reading n_obs from h5ad files: 100%|██████████| 2/2 [00:01<00:00,  1.31file/s]


DistributedAnnDataCollection object with n_obs × n_vars = 3857513 × 38094
  constructed from 386 AnnData objects
    view of obs: 'age', 'assay', 'assay_ontology_term_id', 'biobank', 'brain_region_abbreviation', 'cell_barcode', 'cell_type', 'cell_type_ontology_term_id', 'cohort', 'data_freeze', 'development_stage', 'development_stage_ontology_term_id', 'disease', 'disease_ontology_term_id', 'donor_external_id', 'donor_id', 'experiment', 'expression_doublet', 'extraction', 'frac_contamination', 'hbcac_status', 'imputed_sex', 'mmc_class_name', 'mmc_cluster_name', 'mmc_group_name', 'mmc_neighborhood_name', 'mmc_subclass_name', 'n_counts', 'neuropath_diagnosis', 'num_genes', 'num_genic_reads', 'num_retained_transcripts', 'num_transcripts', 'pct_intergenic', 'pct_intronic', 'pct_mt', 'pct_utr', 'prefix', 'race', 'scpred_class', 'scpred_sub_class', 'self_reported_ethnicity', 'self_reported_ethnicity_ontology_term_id', 'sequencing_assay', 'sex', 'suspension_type', 'tissue', 'tissue_ontology_t

In [43]:
datamodule_var_names_g(datamodule)

soma_joinid
0               A1BG
1           A1BG-AS1
2               A1CF
3                A2M
4            A2M-AS1
            ...     
38089         TRAJ60
38090       FAM90A23
38091    IGHVII-44-2
38092        PRR23D2
38093          TRGJP
Name: gene_name, Length: 38094, dtype: category
Categories (38083, object): ['A1BG', 'A1BG-AS1', 'A1CF', 'A2M', ..., 'ZZEF1', 'ZZZ3', 'hsa-mir-423', 'hsa-mir-1253']

In [44]:
from cellarium.ml.preprocessing import seurat_compute_highly_variable_genes, kotliar_compute_highly_variable_genes
import lightning.pytorch as pl


def cellarium_hvg(
    datamodule: CellariumAnnDataDataModule, 
    n_top_genes: int = 4000, 
    flavor: Literal["seurat_v3", "seurat", "kotliar"] = "seurat",
    batch_key: str | None = None,
) -> tuple[pd.DataFrame, CellariumModule]:
    if flavor not in ["seurat_v3", "seurat", "kotliar"]:
        raise ValueError("Unsupported flavor, choose from ['seurat_v3', 'seurat', 'kotliar']")
    
    if flavor == "seurat_v3":
        module = CellariumModule(
            transforms=[
                Densify(),
                NormalizeTotal(),
                Log1p(),
            ],
            model=HVGSeuratV3(
                var_names_g=datamodule.dadc.var_names,
                n_top_genes=n_top_genes, 
                use_batch_key=batch_key is not None,
                n_batch=1 if batch_key is None else datamodule_obs_nunique(datamodule, batch_key),
            )
        )
    else:
        module = CellariumModule(
            transforms=[
                Densify(),
                NormalizeTotal(),
                Log1p(),
            ],
            model=OnePassMeanVarStd(
                var_names_g=datamodule.dadc.var_names,
                algorithm="shifted_data",
            )
        )

    trainer = pl.Trainer(
        accelerator=ACCELERATOR,
        devices=1,
        logger=False,
        enable_checkpointing=False,
    )

    trainer.fit(module, datamodule)

    if flavor == "seurat_v3":
        hvg_df = module.model._compute_hvg_df(n_top_genes=n_top_genes)
    elif flavor == "seurat":
        hvg_df = seurat_compute_highly_variable_genes(
            var_names_g=datamodule_var_names_g(datamodule),
            mean_g=module.model.mean_g,
            var_g=module.model.var_g,
            n_top_genes=n_top_genes,
        )
    elif flavor == "kotliar":
        hvg_df = kotliar_compute_highly_variable_genes(
            var_names_g=datamodule_var_names_g(datamodule),
            mean_g=module.model.mean_g,
            var_g=module.model.var_g,
            n_top_genes=n_top_genes,
        )

    return hvg_df, module

In [45]:
hvg_df, onepass_module = cellarium_hvg(datamodule, n_top_genes=4000, flavor="seurat")

GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs

  | Name     | Type              | Params | Mode 
-------------------------------------------------------
0 | pipeline | CellariumPipeline | 1      | train
-------------------------------------------------------
1         Trainable params
0         Non-trainable params
1         Total params
0.000     Total estimated model params size (MB)
5         Modules in train mode
0         Modules in eval mode


Epoch 0:  93%|█████████▎| 359/386 [3:04:04<13:50,  0.03it/s]  

ValueError: Expected the number of anndata files (2) to be no more than the max cache size (1).

In [ ]:
def cellarium_pca(datamodule: CellariumAnnDataDataModule, n_components: int = 50, zscore: bool = True) -> CellariumModule:

    module = CellariumModule(
        transforms=[
            Filter(),
            NormalizeTotal(),
            Log1p(),
        ] + ([ZScore()] if zscore else []),

    )
    return module

SyntaxError: incomplete input (3609375515.py, line 4)